# Avito Bot Detection

Задача - по истории событий внутри суточного окна определить, относится ли `cookie_id` к автоматизированному сервису сбора данных.

Целевая переменная:
- `target = 1` - трафик сервисов сбора данных
- `target = 0` - трафик, отнесенный к человеческому

Основная метрика - Precision при Recall ≥ 70%.

В решении:
1. проверяется качество исходных данных
2. события фильтруются по разрешённому окну наблюдения
3. по событиям строятся признаки, описывающие поведение куки
4. модели сравниваются с помощью 5-fold стратифицированной кросс-валидации
5. выбранная модель обучается на всей обучающей выборке и формируется `submission.csv`

Среда: Python 3.14.

In [415]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from metric import precision_at_recall


## 1. Загрузка данных

`train` и `test` содержат по одной строке на `cookie_id`, а `events` содержит историю отдельных событий. Временные поля преобразуются в `datetime`, чтобы далее корректно работать с границами окна.

In [416]:
date_columns = [
    "cookie_created_at",
    "window_start_ts",
    "window_end_ts",
]

train = pd.read_csv("data/train.csv", parse_dates=date_columns)
test = pd.read_csv("data/test.csv", parse_dates=date_columns)
events = pd.read_csv("data/events.csv.gz", compression="gzip", parse_dates=["event_ts"])

In [417]:
train.shape, train.dtypes

((11091, 5),
 cookie_id                       str
 cookie_created_at    datetime64[us]
 window_start_ts      datetime64[us]
 window_end_ts        datetime64[us]
 target                        int64
 dtype: object)

In [418]:
train.head()

,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


In [419]:
test.shape, test.dtypes

((4909, 4),
 cookie_id                       str
 cookie_created_at    datetime64[us]
 window_start_ts      datetime64[us]
 window_end_ts        datetime64[us]
 dtype: object)

In [420]:
test.head()

,cookie_id,cookie_created_at,window_start_ts,window_end_ts
0,ck_315fb710a0e371e7,2026-02-20 08:52:31,2026-04-20,2026-04-21
1,ck_a76ee3b3e3e522fd,2026-01-19 13:42:15,2026-04-20,2026-04-21
2,ck_94c9a4d382689e82,2026-04-19 06:28:37,2026-04-20,2026-04-21
3,ck_8eaf9509ad9462a0,2026-01-19 17:18:06,2026-04-20,2026-04-21
4,ck_9a88a5a989cb5bc6,2025-11-09 17:25:13,2026-04-20,2026-04-21


In [421]:
events.shape, events.dtypes

((328905, 14),
 cookie_id                   str
 event_ts         datetime64[us]
 eid                       int64
 event_name                  str
 platform                    str
 user_agent                  str
 item_id                 float64
 item_category               str
 item_location               str
 seller_type                 str
 search_query                str
 search_page             float64
 pointer_x               float64
 pointer_y               float64
 dtype: object)

In [422]:
events.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN
2,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0
3,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN
4,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,WEB,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN


## 2. Первичный анализ и проверка качества данных

Сначала проверим пропуски, структуру категориальных полей, согласованность `cookie_id` между таблицами и временные ограничения.

In [423]:
events.isna().sum()

cookie_id             0
event_ts              0
eid                   0
event_name            0
platform              0
user_agent            0
item_id          114597
item_category     32920
item_location     23793
seller_type      133853
search_query     228503
search_page      228503
pointer_x        220365
pointer_y        220365
dtype: int64

In [424]:
events["event_name"].value_counts()

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64

In [425]:
nullable_columns = [
    "item_id",
    "item_category",
    "item_location",
    "seller_type",
    "search_query",
    "search_page",
    "pointer_x",
    "pointer_y",
]

event_stat = (
    events.groupby("event_name")[nullable_columns]
    .agg(lambda x: x.notna().mean())
    .reset_index()
)

event_stat

,event_name,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,captcha_shown,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.272200,0.272200
1,contact_chat_open,1.0,0.941224,0.969306,0.915265,0.0,0.0,0.308245,0.308245
2,contact_message_sent,1.0,0.938656,0.971113,0.909120,0.0,0.0,0.322623,0.322623
3,contact_phone_show,1.0,0.941234,0.971456,0.906769,0.0,0.0,0.323259,0.323259
4,favorite_add,1.0,0.936637,0.972072,0.908342,0.0,0.0,0.349467,0.349467
5,item_view,1.0,0.940654,0.969756,0.909806,0.0,0.0,0.326568,0.326568
6,login,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.353439,0.353439
7,photo_swipe,1.0,0.941753,0.968261,0.912041,0.0,0.0,0.341923,0.341923
8,search_results_view,0.0,0.940340,0.969005,0.000000,1.0,1.0,0.330462,0.330462
9,seller_page_view,1.0,0.941562,0.968913,0.911107,0.0,0.0,0.336149,0.336149


Ключевые поля события (`cookie_id`, `event_ts`, `eid`, `event_name`, `platform`, `user_agent`) заполнены полностью. Большая часть пропусков в дополнительных полях имеет структурный характер. Например, `item_id` не используется для `captcha_shown`,
`login` и `search_results_view`, а `search_query` и `search_page` заполнены только для `search_results_view`.

Поэтому массовое удаление строк или заполнение всех строк с `NaN` одним значением считаю некорректным.

In [426]:
group_stat = (
    events.groupby("platform")[nullable_columns]
    .agg(lambda x: x.notna().mean())
    .reset_index()
)

group_stat

,platform,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ANDROID,0.663982,0.902689,0.932128,0.602939,0.297160,0.297160,0.000000,0.000000
1,Android,0.664836,0.902045,0.928669,0.605860,0.294315,0.294315,0.000000,0.000000
2,IOS,0.669024,0.914390,0.938537,0.608049,0.295854,0.295854,0.000000,0.000000
3,WEB,0.641105,0.894333,0.923380,0.583742,0.311150,0.311150,0.581139,0.581139
4,Web,0.643697,0.897164,0.925849,0.585679,0.311248,0.311248,0.588030,0.588030
5,android,0.662065,0.902488,0.931687,0.602552,0.297683,0.297683,0.000000,0.000000
6,desktop,0.642577,0.899693,0.926322,0.585840,0.312721,0.312721,0.586929,0.586929
7,iOS,0.661696,0.912246,0.937912,0.604498,0.302615,0.302615,0.000000,0.000000
8,ios,0.658411,0.904365,0.927661,0.600539,0.300392,0.300392,0.000000,0.000000
9,iphone,0.655862,0.903729,0.932976,0.593712,0.310505,0.310505,0.000000,0.000000


In [427]:
events["platform"].value_counts()

platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64

В `platform` есть варианты, различающиеся регистром (`WEB`, `Web`, `web`, `ANDROID`, `Android`, `android` и т. д.). Они встречаются систематически и в сопоставимых количествах, поэтому не выглядят как ошибки.

Поэтому значения сохраняются в исходном виде, потому что различия в способе записи платформы могут содержать полезный сигнал для классификации.

In [428]:
platform_event_stat = (
    events.groupby(["platform", "event_name"])    
    .agg(
        event_count = ("cookie_id", "size"),
        pointer_x_share = ('pointer_x', lambda x: x.notna().mean()),
        pointer_y_share = ('pointer_y', lambda x: x.notna().mean())
    )
    .reset_index()
)

platform_event_stat.pivot(
    index="platform",
    columns="event_name",
    values="pointer_x_share"
)

event_name,captcha_shown,contact_chat_open,contact_message_sent,contact_phone_show,favorite_add,item_view,login,photo_swipe,search_results_view,seller_page_view
platform,,,,,,,,,,
ANDROID,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Android,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
IOS,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
WEB,0.397037,0.633379,0.626632,0.642366,0.643136,0.561673,0.654776,0.638652,0.575617,0.591174
Web,0.423445,0.639077,0.642857,0.629111,0.671136,0.568364,0.657485,0.636785,0.578200,0.613008
android,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
desktop,0.437754,0.617801,0.681081,0.641152,0.655332,0.577906,0.655889,0.627912,0.568846,0.599185
iOS,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ios,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


Координаты курсора отсутствуют на мобильных вариантах платформ и встречаются только для web и desktop. Поэтому наличие координат далее используется как отдельный поведенческий признак.

In [429]:
train_cookies = set(train["cookie_id"])
test_cookies = set(test["cookie_id"])
event_cookies = set(events["cookie_id"])

print(f"Пересечение train и test: {len(train_cookies & test_cookies)}")
print(f"Количество куки из train без событий: {len(train_cookies - event_cookies)}" )
print(f"Количество куки из test без событий: {len(test_cookies - event_cookies)}") 
print(f"Количество куки из event, которых нет ни в test ни в train: {len(event_cookies - train_cookies - test_cookies)}") 
print(f"Число дубликатов cookie_id в train: {train["cookie_id"].duplicated().sum()}")
print(f"Число дубликатов cookie_id в test: {test["cookie_id"].duplicated().sum()}")

Пересечение train и test: 0
Количество куки из train без событий: 0
Количество куки из test без событий: 0
Количество куки из event, которых нет ни в test ни в train: 0
Число дубликатов cookie_id в train: 0
Число дубликатов cookie_id в test: 0


Проверки согласованности прошли успешно:
- `cookie_id` уникальны внутри `train` и `test`
- `train` и `test` не пересекаются
- каждая cookie имеет события
- в `events` нет cookie, отсутствующих одновременно в `train` и `test`

Значит можно безопасно агрегировать события и связывать таблицы по `cookie_id`.

In [430]:
target_stats = pd.concat([
    train["target"].value_counts(), 
    train["target"].value_counts(normalize=True)
], axis=1).reset_index()

target_stats

,target,count,proportion
0,0,10192,0.918943
1,1,899,0.081057


Положительный класс составляет около 8.1% выборки, классы существенно несбалансированы. При разбиении данных важно сохранять долю положительного класса.

## 3. Проверка временных окон

По условию признаки можно строить только по информации, доступной к моменту окончания суточного окна наблюдения. Значит сначала проверим длительность окон, затем каждому событию сопоставим границы его cookie и исключим события вне допустимого интервала.

In [431]:
train_durations = (
    train["window_end_ts"] - train["window_start_ts"]
).unique()

test_durations = (
    test["window_end_ts"] - test["window_start_ts"]
).unique()

print("Длительности окон в train: \n\n", train_durations)
print()
print("Длительности окон в test: \n\n", test_durations)

Длительности окон в train: 

 <TimedeltaArray>
['1 days']
Length: 1, dtype: timedelta64[us]

Длительности окон в test: 

 <TimedeltaArray>
['1 days']
Length: 1, dtype: timedelta64[us]


In [432]:
window_columns = [
    "cookie_id",
    "window_start_ts",
    "window_end_ts"
]

cookie_windows = pd.concat(
    [
    train[window_columns], 
    test[window_columns],
    ],
    ignore_index=True
)

cookie_windows

,cookie_id,window_start_ts,window_end_ts
0,ck_54a059eb7d3ea68b,2026-04-06,2026-04-07
1,ck_7e4de46eeab82974,2026-04-06,2026-04-07
2,ck_9320229ef6304522,2026-04-06,2026-04-07
3,ck_30ccd25bc1714ed9,2026-04-06,2026-04-07
4,ck_a77c5f05948cdeef,2026-04-06,2026-04-07
...,...,...,...
15995,ck_8b772caed8f1541c,2026-04-26,2026-04-27
15996,ck_5403c76fe68a08ba,2026-04-26,2026-04-27
15997,ck_3e6833574edf1736,2026-04-26,2026-04-27
15998,ck_e527fbb9450a4581,2026-04-26,2026-04-27


In [433]:
events_windows = pd.merge(events, cookie_windows, on="cookie_id", how="left", validate="many_to_one")

events_windows.head(2)

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN,2026-04-26,2026-04-27
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN,2026-04-20,2026-04-21


In [434]:
window_mask = (
    (events_windows["window_start_ts"] <= events_windows["event_ts"]) 
    & (events_windows["event_ts"] < events_windows["window_end_ts"])
)

print(f"Событий вне окна наблюдения: {(~window_mask).sum()}")

Событий вне окна наблюдения: 40779


In [435]:
before_window = events_windows["event_ts"] < events_windows["window_start_ts"]
after_window = events_windows["event_ts"] >= events_windows["window_end_ts"]

print(f"Событий до начала окна {before_window.sum()}")
print(f"Событий после конца окна {after_window.sum()}")

Событий до начала окна 0
Событий после конца окна 40779


In [436]:
events_in_window = events_windows.loc[window_mask].copy()

In [437]:
filtered_cookies = set(events_in_window["cookie_id"])
missing_after_filter = (train_cookies | test_cookies) - filtered_cookies

print(f"Количество куки без событий после фильтрации: {len(missing_after_filter)}")

Количество куки без событий после фильтрации: 0


Обнаружено 40 779 событий за пределами разрешённого окна. Для построения признаков используются только события, удовлетворяющие условию

`window_start_ts <= event_ts < window_end_ts`.

После фильтрации у всех cookie остаётся хотя бы одно событие.

In [438]:
train_created_in_window = (
    (train["cookie_created_at"] >= train["window_start_ts"])
    & (train["cookie_created_at"] < train["window_end_ts"])
)

test_created_in_window = (
    (test["cookie_created_at"] >= test["window_start_ts"])
    & (test["cookie_created_at"] < test["window_end_ts"])
)

print(f"Куки в train создано внутри окна: {train_created_in_window.sum()}")
print(f"Куки в test создано внутри окна: {test_created_in_window.sum()}")
print(f"Куки в train создано после окончания окна: {(train["cookie_created_at"] >= train["window_end_ts"]).sum()}")
print(f"Куки в test создано после окончания окна: {(test["cookie_created_at"] >= test["window_end_ts"]).sum()}")

Куки в train создано внутри окна: 0
Куки в test создано внутри окна: 0
Куки в train создано после окончания окна: 0
Куки в test создано после окончания окна: 0


Все куки были созданы до начала окна наблюдения, поэтому возраст cookie к началу окна можно использовать как корректный признак.

## 4. Построение признаков

Модель должна выдавать один score на `cookie_id`, тогда как `events` содержит много строк на одну куки, поэтому события агрегируются до уровня `cookie_id`.

В набор признаков входят:
- общее количество событий
- количество и доля каждого типа события
- число уникальных `item_id` и `item_category`
- число уникальных значений `platform`, `item_location`, `seller_type` и `search_query`
- длительность активного периода
- среднее, медианное, минимальное и максимальное время между соседними событиями, а также стандартное отклонение интервалов
- возраст куки к началу окна наблюдения
- количество и доля событий для каждого значения `platform`
- количество и доля событий, для которых доступны координаты курсора

In [439]:
features = (
    events_in_window.groupby("cookie_id")
    .agg(
        event_count=("event_name", "count"),
        unique_event_types=("event_name", "nunique"),
        unique_items=("item_id", "nunique"),
        unique_categories=("item_category", "nunique")
    )
    .reset_index()
)

features.head()

,cookie_id,event_count,unique_event_types,unique_items,unique_categories
0,ck_00013fdfa0bd37dd,15,6,9,3
1,ck_000c95f1408dcb00,16,4,9,2
2,ck_000e8c52636e3bec,34,7,20,4
3,ck_0010e31baa4a1fb7,16,6,8,3
4,ck_0010ec3874fb5378,74,6,31,2


In [440]:
event_type_counts = (
    events_in_window
    .groupby(["cookie_id", "event_name"])
    .size()
    .unstack(fill_value=0)
    .add_prefix("event_")
    .add_suffix("_count")
    .reset_index()
)

event_type_counts.head()

event_name,cookie_id,event_contact_chat_open_count,event_contact_message_sent_count,event_contact_phone_show_count,event_favorite_add_count,event_item_view_count,event_login_count,event_photo_swipe_count,event_search_results_view_count,event_seller_page_view_count
0,ck_00013fdfa0bd37dd,2,0,0,1,4,2,3,3,0
1,ck_000c95f1408dcb00,0,0,0,0,11,0,1,3,1
2,ck_000e8c52636e3bec,0,0,1,4,11,1,5,9,3
3,ck_0010e31baa4a1fb7,0,0,0,2,7,1,1,4,1
4,ck_0010ec3874fb5378,0,0,0,10,25,1,14,20,4


In [441]:
features = features.merge(
    event_type_counts,
    on="cookie_id",
    how="left",
    validate="one_to_one"
)

features.head()

,cookie_id,event_count,unique_event_types,unique_items,unique_categories,event_contact_chat_open_count,event_contact_message_sent_count,event_contact_phone_show_count,event_favorite_add_count,event_item_view_count,event_login_count,event_photo_swipe_count,event_search_results_view_count,event_seller_page_view_count
0,ck_00013fdfa0bd37dd,15,6,9,3,2,0,0,1,4,2,3,3,0
1,ck_000c95f1408dcb00,16,4,9,2,0,0,0,0,11,0,1,3,1
2,ck_000e8c52636e3bec,34,7,20,4,0,0,1,4,11,1,5,9,3
3,ck_0010e31baa4a1fb7,16,6,8,3,0,0,0,2,7,1,1,4,1
4,ck_0010ec3874fb5378,74,6,31,2,0,0,0,10,25,1,14,20,4


In [442]:
time_features = (
    events_in_window.groupby("cookie_id")
    .agg(
        first_event_ts=("event_ts", "min"),
        last_event_ts=("event_ts", "max")
    )
    .reset_index()
)

time_features.head()

,cookie_id,first_event_ts,last_event_ts
0,ck_00013fdfa0bd37dd,2026-04-24 18:41:16,2026-04-24 20:48:46
1,ck_000c95f1408dcb00,2026-04-19 14:49:54,2026-04-19 23:11:51
2,ck_000e8c52636e3bec,2026-04-07 06:26:00,2026-04-07 08:17:21
3,ck_0010e31baa4a1fb7,2026-04-15 11:53:18,2026-04-15 23:39:34
4,ck_0010ec3874fb5378,2026-04-06 02:08:23,2026-04-06 17:59:27


In [443]:
time_features["active_span_seconds"] = (
    time_features["last_event_ts"] - time_features["first_event_ts"]
).dt.total_seconds()

time_features.head()

,cookie_id,first_event_ts,last_event_ts,active_span_seconds
0,ck_00013fdfa0bd37dd,2026-04-24 18:41:16,2026-04-24 20:48:46,7650.0
1,ck_000c95f1408dcb00,2026-04-19 14:49:54,2026-04-19 23:11:51,30117.0
2,ck_000e8c52636e3bec,2026-04-07 06:26:00,2026-04-07 08:17:21,6681.0
3,ck_0010e31baa4a1fb7,2026-04-15 11:53:18,2026-04-15 23:39:34,42376.0
4,ck_0010ec3874fb5378,2026-04-06 02:08:23,2026-04-06 17:59:27,57064.0


In [444]:
events_sorted = events_in_window.sort_values(
    ["cookie_id", "event_ts"]
).copy()

events_sorted["event_gap_seconds"] = (
    events_sorted.groupby("cookie_id")["event_ts"]
    .diff()
    .dt.total_seconds()
)

events_sorted.head(2)

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts,event_gap_seconds
67286,ck_00013fdfa0bd37dd,2026-04-24 18:41:16,210,photo_swipe,ANDROID,Mozilla/5.0 (Linux; Android 12; Pixel 6) Apple...,1012349.0,rabota,moskva,private,NaN,NaN,NaN,NaN,2026-04-24,2026-04-25,NaN
72380,ck_00013fdfa0bd37dd,2026-04-24 19:16:53,200,item_view,android,Mozilla/5.0 (Linux; Android 12; Pixel 6) Apple...,1054355.0,rabota,moskva,private,NaN,NaN,NaN,NaN,2026-04-24,2026-04-25,2137.0


In [445]:
gap_features = (
    events_sorted.groupby("cookie_id")
    .agg(
        gap_mean=("event_gap_seconds", "mean"),
        gap_median=("event_gap_seconds", "median"),
        gap_std=("event_gap_seconds", "std"),
        gap_min=("event_gap_seconds", "min"),
        gap_max=("event_gap_seconds", "max"),
    )
    .reset_index()
)

gap_features.head()

,cookie_id,gap_mean,gap_median,gap_std,gap_min,gap_max
0,ck_00013fdfa0bd37dd,546.428571,148.5,936.612777,1.0,2880.0
1,ck_000c95f1408dcb00,2007.800000,58.0,3080.553503,17.0,7733.0
2,ck_000e8c52636e3bec,202.454545,23.0,774.305983,0.0,4257.0
3,ck_0010e31baa4a1fb7,2825.066667,334.0,3107.706813,10.0,8243.0
4,ck_0010ec3874fb5378,781.698630,32.0,1984.404925,0.0,8567.0


In [446]:
features = (
    features
    .merge(
        time_features[["cookie_id", "active_span_seconds"]],
        on="cookie_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        gap_features,
        on="cookie_id",
        how="left",
        validate="one_to_one"
    )
)  

features.head()

,cookie_id,event_count,unique_event_types,unique_items,unique_categories,event_contact_chat_open_count,event_contact_message_sent_count,event_contact_phone_show_count,event_favorite_add_count,event_item_view_count,event_login_count,event_photo_swipe_count,event_search_results_view_count,event_seller_page_view_count,active_span_seconds,gap_mean,gap_median,gap_std,gap_min,gap_max
0,ck_00013fdfa0bd37dd,15,6,9,3,2,0,0,1,4,2,3,3,0,7650.0,546.428571,148.5,936.612777,1.0,2880.0
1,ck_000c95f1408dcb00,16,4,9,2,0,0,0,0,11,0,1,3,1,30117.0,2007.800000,58.0,3080.553503,17.0,7733.0
2,ck_000e8c52636e3bec,34,7,20,4,0,0,1,4,11,1,5,9,3,6681.0,202.454545,23.0,774.305983,0.0,4257.0
3,ck_0010e31baa4a1fb7,16,6,8,3,0,0,0,2,7,1,1,4,1,42376.0,2825.066667,334.0,3107.706813,10.0,8243.0
4,ck_0010ec3874fb5378,74,6,31,2,0,0,0,10,25,1,14,20,4,57064.0,781.698630,32.0,1984.404925,0.0,8567.0


In [447]:
gap_missing = gap_features.isna().sum()
gap_missing[gap_missing > 0]

gap_mean      351
gap_median    351
gap_std       933
gap_min       351
gap_max       351
dtype: int64

In [448]:
features.loc[
    features["gap_std"].isna(),
    "event_count"
].value_counts().sort_index()

event_count
1    351
2    582
Name: count, dtype: int64

Пропуски во временных статистиках ожидаемы, потому что они связаны с небольшим числом событий у части куки. У 351 куки присутствует только одно событие, поэтому интервалы между событиями для них рассчитать нельзя. По этой причине `gap_mean`, `gap_median`, `gap_min` и `gap_max` содержат 351 пропуск. 

Ещё 582 куки имеют по два события, то есть только один интервал, а этого, в свою очередь, недостаточно для расчёта стандартного отклонения, поэтому в `gap_std` дополнительно появляются 582 пропуска. Всего `gap_std` содержит 351 + 582 = 933 пропуска.

In [449]:
cookie_meta_columns = [
    "cookie_id",
    "cookie_created_at",
    "window_start_ts",
]

cookie_meta = pd.concat(
    [
        train[cookie_meta_columns],
        test[cookie_meta_columns]
    ],
    ignore_index=True,
)

cookie_meta["cookie_age_hours"] = (
    cookie_meta["window_start_ts"] - cookie_meta["cookie_created_at"]
).dt.total_seconds() / 3600

cookie_meta.head()

,cookie_id,cookie_created_at,window_start_ts,cookie_age_hours
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,3254.488611
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,4669.826667
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,791.866111
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,13.122222
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2276.090278


In [450]:
features = features.merge(
    cookie_meta[["cookie_id", "cookie_age_hours"]],
    on="cookie_id",
    how="left",
    validate="one_to_one",
)

features.head()

,cookie_id,event_count,unique_event_types,unique_items,unique_categories,event_contact_chat_open_count,event_contact_message_sent_count,event_contact_phone_show_count,event_favorite_add_count,event_item_view_count,...,event_photo_swipe_count,event_search_results_view_count,event_seller_page_view_count,active_span_seconds,gap_mean,gap_median,gap_std,gap_min,gap_max,cookie_age_hours
0,ck_00013fdfa0bd37dd,15,6,9,3,2,0,0,1,4,...,3,3,0,7650.0,546.428571,148.5,936.612777,1.0,2880.0,4709.170278
1,ck_000c95f1408dcb00,16,4,9,2,0,0,0,0,11,...,1,3,1,30117.0,2007.800000,58.0,3080.553503,17.0,7733.0,5591.050556
2,ck_000e8c52636e3bec,34,7,20,4,0,0,1,4,11,...,5,9,3,6681.0,202.454545,23.0,774.305983,0.0,4257.0,611.523611
3,ck_0010e31baa4a1fb7,16,6,8,3,0,0,0,2,7,...,1,4,1,42376.0,2825.066667,334.0,3107.706813,10.0,8243.0,1985.161667
4,ck_0010ec3874fb5378,74,6,31,2,0,0,0,10,25,...,14,20,4,57064.0,781.698630,32.0,1984.404925,0.0,8567.0,2457.485000


In [451]:
diversity_features = (
    events_in_window.groupby("cookie_id")
    .agg(
        unique_platform=("platform", "nunique"),
        unique_user_agents=("user_agent", "nunique"),
        unique_locations=("item_location", "nunique"),
        unique_seller_types=("seller_type", "nunique"),
        unique_search_queries=("search_query", "nunique"),
    )
    .reset_index()
)

diversity_features.head()

,cookie_id,unique_platform,unique_user_agents,unique_locations,unique_seller_types,unique_search_queries
0,ck_00013fdfa0bd37dd,3,1,4,2,3
1,ck_000c95f1408dcb00,4,1,8,2,3
2,ck_000e8c52636e3bec,4,1,12,2,8
3,ck_0010e31baa4a1fb7,4,1,4,2,4
4,ck_0010ec3874fb5378,4,1,19,2,8


In [452]:
features = features.merge(
    diversity_features,
    on="cookie_id",
    how="left",
    validate="one_to_one",
)

features.head()

,cookie_id,event_count,unique_event_types,unique_items,unique_categories,event_contact_chat_open_count,event_contact_message_sent_count,event_contact_phone_show_count,event_favorite_add_count,event_item_view_count,...,gap_median,gap_std,gap_min,gap_max,cookie_age_hours,unique_platform,unique_user_agents,unique_locations,unique_seller_types,unique_search_queries
0,ck_00013fdfa0bd37dd,15,6,9,3,2,0,0,1,4,...,148.5,936.612777,1.0,2880.0,4709.170278,3,1,4,2,3
1,ck_000c95f1408dcb00,16,4,9,2,0,0,0,0,11,...,58.0,3080.553503,17.0,7733.0,5591.050556,4,1,8,2,3
2,ck_000e8c52636e3bec,34,7,20,4,0,0,1,4,11,...,23.0,774.305983,0.0,4257.0,611.523611,4,1,12,2,8
3,ck_0010e31baa4a1fb7,16,6,8,3,0,0,0,2,7,...,334.0,3107.706813,10.0,8243.0,1985.161667,4,1,4,2,4
4,ck_0010ec3874fb5378,74,6,31,2,0,0,0,10,25,...,32.0,1984.404925,0.0,8567.0,2457.485000,4,1,19,2,8


In [453]:
event_count_columns = [
    column
    for column in features.columns
    if column.startswith("event_")
    and column.endswith("_count")
    and column != "event_count"
]

for column in event_count_columns:
    share_column = column.replace("_count", "_share")
    features[share_column] = features[column] / features["event_count"]

features.head()

,cookie_id,event_count,unique_event_types,unique_items,unique_categories,event_contact_chat_open_count,event_contact_message_sent_count,event_contact_phone_show_count,event_favorite_add_count,event_item_view_count,...,unique_search_queries,event_contact_chat_open_share,event_contact_message_sent_share,event_contact_phone_show_share,event_favorite_add_share,event_item_view_share,event_login_share,event_photo_swipe_share,event_search_results_view_share,event_seller_page_view_share
0,ck_00013fdfa0bd37dd,15,6,9,3,2,0,0,1,4,...,3,0.133333,0.0,0.000000,0.066667,0.266667,0.133333,0.200000,0.200000,0.000000
1,ck_000c95f1408dcb00,16,4,9,2,0,0,0,0,11,...,3,0.000000,0.0,0.000000,0.000000,0.687500,0.000000,0.062500,0.187500,0.062500
2,ck_000e8c52636e3bec,34,7,20,4,0,0,1,4,11,...,8,0.000000,0.0,0.029412,0.117647,0.323529,0.029412,0.147059,0.264706,0.088235
3,ck_0010e31baa4a1fb7,16,6,8,3,0,0,0,2,7,...,4,0.000000,0.0,0.000000,0.125000,0.437500,0.062500,0.062500,0.250000,0.062500
4,ck_0010ec3874fb5378,74,6,31,2,0,0,0,10,25,...,8,0.000000,0.0,0.000000,0.135135,0.337838,0.013514,0.189189,0.270270,0.054054


In [454]:
platform_counts = (
    events_in_window
    .groupby(["cookie_id", "platform"])
    .size()
    .unstack(fill_value=0)
    .add_prefix("platform_")
    .add_suffix("_count")
    .reset_index()
)

features = features.merge(
    platform_counts,
    on="cookie_id",
    how="left",
    validate="one_to_one"
)

features.head()

,cookie_id,event_count,unique_event_types,unique_items,unique_categories,event_contact_chat_open_count,event_contact_message_sent_count,event_contact_phone_show_count,event_favorite_add_count,event_item_view_count,...,platform_Android_count,platform_IOS_count,platform_WEB_count,platform_Web_count,platform_android_count,platform_desktop_count,platform_iOS_count,platform_ios_count,platform_iphone_count,platform_web_count
0,ck_00013fdfa0bd37dd,15,6,9,3,2,0,0,1,4,...,3,0,0,0,6,0,0,0,0,0
1,ck_000c95f1408dcb00,16,4,9,2,0,0,0,0,11,...,0,0,4,8,0,1,0,0,0,3
2,ck_000e8c52636e3bec,34,7,20,4,0,0,1,4,11,...,0,0,9,5,0,14,0,0,0,6
3,ck_0010e31baa4a1fb7,16,6,8,3,0,0,0,2,7,...,0,0,2,5,0,5,0,0,0,4
4,ck_0010ec3874fb5378,74,6,31,2,0,0,0,10,25,...,0,0,18,20,0,17,0,0,0,19


In [455]:
platform_count_columns = [
    column
    for column in features.columns
    if column.startswith("platform_")
    and column.endswith("_count")
]

for column in platform_count_columns:
    share_column = column.replace("_count", "_share")
    features[share_column] = features[column] / features["event_count"]

features.head()

,cookie_id,event_count,unique_event_types,unique_items,unique_categories,event_contact_chat_open_count,event_contact_message_sent_count,event_contact_phone_show_count,event_favorite_add_count,event_item_view_count,...,platform_Android_share,platform_IOS_share,platform_WEB_share,platform_Web_share,platform_android_share,platform_desktop_share,platform_iOS_share,platform_ios_share,platform_iphone_share,platform_web_share
0,ck_00013fdfa0bd37dd,15,6,9,3,2,0,0,1,4,...,0.2,0.0,0.000000,0.000000,0.4,0.000000,0.0,0.0,0.0,0.000000
1,ck_000c95f1408dcb00,16,4,9,2,0,0,0,0,11,...,0.0,0.0,0.250000,0.500000,0.0,0.062500,0.0,0.0,0.0,0.187500
2,ck_000e8c52636e3bec,34,7,20,4,0,0,1,4,11,...,0.0,0.0,0.264706,0.147059,0.0,0.411765,0.0,0.0,0.0,0.176471
3,ck_0010e31baa4a1fb7,16,6,8,3,0,0,0,2,7,...,0.0,0.0,0.125000,0.312500,0.0,0.312500,0.0,0.0,0.0,0.250000
4,ck_0010ec3874fb5378,74,6,31,2,0,0,0,10,25,...,0.0,0.0,0.243243,0.270270,0.0,0.229730,0.0,0.0,0.0,0.256757


In [456]:
pointer_features = (
    events_in_window.assign(
        has_pointer=events_in_window["pointer_x"].notna()
    )
    .groupby("cookie_id")
    .agg(
        pointer_event_count=("has_pointer", "sum"),
        pointer_event_share=("has_pointer", "mean"),
    )
    .reset_index()
)

features = features.merge(
    pointer_features,
    on="cookie_id",
    how="left",
    validate="one_to_one",
)

features.head()

,cookie_id,event_count,unique_event_types,unique_items,unique_categories,event_contact_chat_open_count,event_contact_message_sent_count,event_contact_phone_show_count,event_favorite_add_count,event_item_view_count,...,platform_WEB_share,platform_Web_share,platform_android_share,platform_desktop_share,platform_iOS_share,platform_ios_share,platform_iphone_share,platform_web_share,pointer_event_count,pointer_event_share
0,ck_00013fdfa0bd37dd,15,6,9,3,2,0,0,1,4,...,0.000000,0.000000,0.4,0.000000,0.0,0.0,0.0,0.000000,0,0.000000
1,ck_000c95f1408dcb00,16,4,9,2,0,0,0,0,11,...,0.250000,0.500000,0.0,0.062500,0.0,0.0,0.0,0.187500,16,1.000000
2,ck_000e8c52636e3bec,34,7,20,4,0,0,1,4,11,...,0.264706,0.147059,0.0,0.411765,0.0,0.0,0.0,0.176471,31,0.911765
3,ck_0010e31baa4a1fb7,16,6,8,3,0,0,0,2,7,...,0.125000,0.312500,0.0,0.312500,0.0,0.0,0.0,0.250000,16,1.000000
4,ck_0010ec3874fb5378,74,6,31,2,0,0,0,10,25,...,0.243243,0.270270,0.0,0.229730,0.0,0.0,0.0,0.256757,0,0.000000


In [457]:
missing_by_feature = features.isna().sum()
missing_by_feature[missing_by_feature > 0]

gap_mean      351
gap_median    351
gap_std       933
gap_min       351
gap_max       351
dtype: int64

In [458]:
features.shape

(16000, 59)

## 5. Подготовка выборок для моделирования

После построения общей таблицы признаков она разделяется обратно на обучающую и тестовую части по `cookie_id`. Сам `cookie_id` в модель не передаётся, так как является идентификатором, а не поведенческим признаком.

In [459]:
train_features = train[["cookie_id", "target"]].merge(
    features,
    on="cookie_id",
    how="left",
    validate="one_to_one"
)

train_features.shape

(11091, 60)

In [460]:
test_features = test[["cookie_id"]].merge(
    features,
    on="cookie_id",
    how="left",
    validate="one_to_one",
)

test_features.shape

(4909, 59)

In [461]:
X = train_features.drop(columns=["cookie_id", "target"])
y = train_features["target"]

X_test = test_features.drop(columns=["cookie_id"])

## 6. Валидация и baseline

Из-за дисбаланса классов используется 5-fold `StratifiedKFold`. Каждый фолд сохраняет примерно одинаковую долю положительного класса, `random_state=42` обеспечивает воспроизводимость.

Основная метрика считается функцией `precision_at_recall` из предоставленного `metric.py`, дополнительно для диагностики считаются PR-AUC и ROC-AUC.

В качестве baseline выбран `HistGradientBoostingClassifier`, он подходит для нелинейных зависимостей в табличных данных и умеет работать с оставленными `NaN`.

In [462]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

In [463]:
cv_scores = []
cv_pr_auc = []
cv_roc_auc = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    X_train_fold = X.iloc[train_idx]
    X_valid_fold = X.iloc[valid_idx]

    y_train_fold = y.iloc[train_idx]
    y_valid_fold = y.iloc[valid_idx]

    model = HistGradientBoostingClassifier(
        max_iter=200,
        random_state=42,
    )

    model.fit(X_train_fold, y_train_fold)

    valid_score = model.predict_proba(X_valid_fold)[:, 1]

    score = precision_at_recall(y_valid_fold, valid_score)
    pr_auc = average_precision_score(y_valid_fold, valid_score)
    roc_auc = roc_auc_score(y_valid_fold, valid_score)

    cv_scores.append(score)
    cv_pr_auc.append(pr_auc)
    cv_roc_auc.append(roc_auc)

    print(
        f"Fold {fold}: "
        f"P@R70={score:.4f}, "
        f"PR-AUC={pr_auc:.4f}, "
        f"ROC-AUC={roc_auc:.4f}"
    )

Fold 1: P@R70=0.4684, PR-AUC=0.6922, ROC-AUC=0.8970
Fold 2: P@R70=0.5101, PR-AUC=0.6790, ROC-AUC=0.8843
Fold 3: P@R70=0.5207, PR-AUC=0.6851, ROC-AUC=0.8963
Fold 4: P@R70=0.4242, PR-AUC=0.6489, ROC-AUC=0.8633
Fold 5: P@R70=0.3728, PR-AUC=0.6388, ROC-AUC=0.8626


In [464]:
print(f"P@R70: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")
print(f"PR-AUC: {np.mean(cv_pr_auc):.4f} ± {np.std(cv_pr_auc):.4f}")
print(f"ROC-AUC: {np.mean(cv_roc_auc):.4f} ± {np.std(cv_roc_auc):.4f}")

P@R70: 0.4592 ± 0.0550
PR-AUC: 0.6688 ± 0.0210
ROC-AUC: 0.8807 ± 0.0152


Baseline показывает средний `P@R70 = 0.4592 ± 0.0550` на пяти фолдах. Дополнительные метрики: `PR-AUC = 0.6688 ± 0.0210`, `ROC-AUC = 0.8807 ± 0.0152`.

Разброс основной метрики между фолдами заметен, поэтому для сравнения моделей используется одинаковая схема 5-fold CV.

## 7. Сравнение моделей

Чтобы проверить выбор baseline-модели, сравниваются несколько семейств алгоритмов на одинаковых фолдах и по одной основной метрике:
- Logistic Regression - линейная модель
- Random Forest и Extra Trees - ансамбли независимых деревьев
- HistGradientBoosting - градиентный бустинг

Для моделей, которые не работают с пропусками напрямую, используется медианная импутация, для Logistic Regression дополнительно применяется масштабирование.

In [465]:
models = {
    "logistic_regression": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        ),
    ),

    "random_forest": make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            n_jobs=-1,
            random_state=42,
        ),
    ),

    "extra_trees": make_pipeline(
        SimpleImputer(strategy="median"),
        ExtraTreesClassifier(
            n_estimators=300,
            class_weight="balanced",
            n_jobs=-1,
            random_state=42,
        ),
    ),

    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_iter=200,
        random_state=42,
    ),
}

In [466]:
model_comparison = []

for model_name, model in models.items():
    fold_scores = []

    for train_idx, valid_idx in cv.split(X, y):
        X_train_fold = X.iloc[train_idx]
        X_valid_fold = X.iloc[valid_idx]
        y_train_fold = y.iloc[train_idx]
        y_valid_fold = y.iloc[valid_idx]

        fold_model = clone(model)
        fold_model.fit(X_train_fold, y_train_fold)

        valid_score = fold_model.predict_proba(X_valid_fold)[:, 1]

        fold_scores.append(
            precision_at_recall(y_valid_fold, valid_score)
        )

    model_comparison.append({
        "model": model_name,
        "P@R70_mean": np.mean(fold_scores),
        "P@R70_std": np.std(fold_scores),
    })

model_comparison = (
    pd.DataFrame(model_comparison)
    .sort_values("P@R70_mean", ascending=False)
    .reset_index(drop=True)
)

model_comparison

,model,P@R70_mean,P@R70_std
0,hist_gradient_boosting,0.459242,0.055037
1,random_forest,0.412163,0.040529
2,extra_trees,0.327986,0.069909
3,logistic_regression,0.273903,0.043344


Лучший средний `P@R70` равный 0.4592 показал `HistGradientBoostingClassifier`, поэтому дальнейший подбор гиперпараметров проводится только для этой модели.

## 8. Настройка HistGradientBoosting

После выбора `HistGradientBoostingClassifier` проверим несколько вариантов его настройки. Для этого сравниваются базовая конфигурация, уменьшение числа листьев, L2-регуляризация и меньший `learning_rate`.

Полный перебор гиперпараметров не проводится. Нужно проверить несколько разумных конфигураций и выбрать лучшую по средней целевой метрике на той же 5-fold кросс-валидации

In [467]:
model_configs = {
    "baseline": {
        "max_iter": 200,
        "random_state": 42,
    },

    "smaller_leaves": {
        "max_iter": 300,
        "max_leaf_nodes": 15,
        "random_state": 42,
    },

    "regularized": {
        "max_iter": 300,
        "l2_regularization": 1.0,
        "random_state": 42,
    },

    "slow_learning": {
        "learning_rate": 0.05,
        "max_iter": 400,
        "l2_regularization": 1.0,
        "random_state": 42,
    },
}

In [468]:
model_results = []

for model_name, params in model_configs.items():
    fold_scores = []

    for train_idx, valid_idx in cv.split(X, y):
        X_train_fold = X.iloc[train_idx]
        X_valid_fold = X.iloc[valid_idx]
        y_train_fold = y.iloc[train_idx]
        y_valid_fold = y.iloc[valid_idx]

        model = HistGradientBoostingClassifier(**params)
        model.fit(X_train_fold, y_train_fold)

        valid_score = model.predict_proba(X_valid_fold)[:, 1]

        fold_scores.append(
            precision_at_recall(y_valid_fold, valid_score)
        )

    model_results.append({
        "model": model_name,
        "P@R70_mean": np.mean(fold_scores),
        "P@R70_std": np.std(fold_scores),
    })

model_results = (
    pd.DataFrame(model_results)
    .sort_values("P@R70_mean", ascending=False)
    .reset_index(drop=True)
)

model_results

,model,P@R70_mean,P@R70_std
0,smaller_leaves,0.480789,0.075771
1,baseline,0.459242,0.055037
2,slow_learning,0.450102,0.078210
3,regularized,0.422220,0.056991


Лучший средний результат среди проверенных конфигураций получен при `max_leaf_nodes=15` и `max_iter=300`: 

`P@R70 = 0.4808 ± 0.0758`.

Это выше baseline на примерно 0.0215 абсолютного Precision, при этом разброс между фолдами вырос. Эта конфигурация будет использоваться как финальная.

## 9. Финальная модель и submission

После выбора конфигурации модель обучается заново на всей размеченной выборке.

In [469]:
final_model = HistGradientBoostingClassifier(
    max_iter=300,
    max_leaf_nodes=15,
    random_state=42,
)

final_model.fit(X, y)

,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",300
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",15
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""Categorical"" and ""Enum"" are considered to be categorical features. The input must be a dataframe that is supported by narwhals (or supports it): :func:`narwhals.from_native` must work. This is the case, for instance, for pandas and polars DataFrames.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing val

In [470]:
test_score = final_model.predict_proba(X_test)[:, 1]

In [471]:
submission = test_features[["cookie_id"]].copy()
submission["score"] = test_score

submission.head()

,cookie_id,score
0,ck_315fb710a0e371e7,0.009685
1,ck_a76ee3b3e3e522fd,0.134572
2,ck_94c9a4d382689e82,0.023966
3,ck_8eaf9509ad9462a0,0.009507
4,ck_9a88a5a989cb5bc6,0.007549


In [472]:
print("Размер submission:", submission.shape)
print("Пропуски:")
print(submission.isna().sum())

print("Дубликаты cookie_id:", submission["cookie_id"].duplicated().sum())
print("Минимальный score:", submission["score"].min())
print("Максимальный score:", submission["score"].max())

Размер submission: (4909, 2)
Пропуски:
cookie_id    0
score        0
dtype: int64
Дубликаты cookie_id: 0
Минимальный score: 0.0011962583754303656
Максимальный score: 0.9994221360608763


In [473]:
submission.to_csv("submission.csv", index=False)